# Gemma-Titans Training on TPU with Kauldron

This notebook demonstrates the complete pipeline for training and using the **Gemma-Titans** hybrid model on a Google Cloud TPU v5e using the `Kauldron` framework (DeepMind's standard training library).

### Steps included:
1. **Initialization:** Loading base Gemma weights and randomly initializing Titans NLTM modules using `SkipTitans`.
2. **Pre-training (CPT):** Training *only* the Titans memory layers using `kd.optim.partial_updates` to avoid TPU OOM.
3. **Save/Load:** Splitting the PyTree to save only the fine-tuned memory weights.
4. **Dialogue Inference:** Running the model and updating the internal memory cache dynamically.

In [ ]:
# 0. Environment Setup
!pip install -q kauldron flax optax seqio

# Clone the gemma repository if not already present
!git clone https://github.com/google-deepmind/gemma.git || true
!pip install -q ./gemma

# Ensure our local modules are in the Python path
import sys
import os
sys.path.append(os.getcwd())

In [ ]:
import jax
import jax.numpy as jnp
import optax
from kauldron import kd
import orbax.checkpoint as ocp

# Gemma imports
from gemma import gm
from gemma.gm.nn import _config

# Our custom Titans integration
from gemma_titans import Gemma3_1B_Titans, TitansLayerCache
from titans_ckpts import SkipTitans
import titans_tree_utils

print(f"JAX Backend: {jax.default_backend()}")
print(f"Devices: {jax.devices()}")

# Prevent JAX from allocating 100% of TPU memory instantly (good for debugging)
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"


## 1. Initialization & 2. Pre-training (CPT)

We use `Kauldron` to orchestrate the training. 
- `SkipTitans` handles loading Gemma weights while keeping Titans randomized.
- `kd.optim.partial_updates` ensures XLA only builds backprop graphs for memory parameters.

In [ ]:
# Define the model configuration (e.g., 1B or 7B)
gemma_config = _config.Gemma3_1B_Config()

# Initialize model
model = Gemma3_1B_Titans(
    config=gemma_config,
    return_last_only=True, # Standard for classification/autoregressive modeling
)

# Dummy dataset (replace with kd.data.SeqIO or kd.data.InMemoryDataset)
def dummy_dataset():
    while True:
        yield {
            "tokens": jnp.ones((2, 128), dtype=jnp.int32),
            "target": jnp.ones((2,), dtype=jnp.int32),
            "loss_mask": jnp.ones((2,), dtype=jnp.float32),
        }

# Configure Trainer
trainer = kd.train.Trainer(
    seed=42,
    workdir='./titans_workdir',
    train_ds=dummy_dataset(),
    model=model,
    
    # --- THE MAGIC: SkipTitans ---
    # Loads base weights from checkpoint, leaves Titans layers as random init
    init_transform=SkipTitans(
        wrapped=gm.ckpts.LoadCheckpoint(
            path=gm.ckpts.CheckpointPath.GEMMA3_1B_IT,
        )
    ),
    
    num_train_steps=500,
    train_losses={
        "xentropy": kd.losses.SoftmaxCrossEntropyWithIntLabels(
            logits="preds.logits",
            labels="batch.target",
            mask="batch.loss_mask",
        ),
    },
    
    # --- PREVENT TPU OOM ---
    # Mask out the Gemma weights so gradients are ONLY computed for memory
    optimizer=kd.optim.partial_updates(
        optax.adam(learning_rate=1e-4),
        mask=kd.optim.select(["memory", "memory_gate"]),
    ),      
    checkpointer=kd.ckpts.Checkpointer(save_interval_steps=100),
    
    # Sharding for TPU / TPU Pods
    sharding=kd.sharding.ShardingStrategy(),
)

print("Trainer initialized. Starting compilation and training...")

# Uncomment to run the actual training loop:
# state, aux = trainer.train()

## 3. Save / Load Trained Weights
We don't want to save the entire 1B model, just our new memory projections. We utilize the `split_titans_params` utility.

In [ ]:
def save_titans_weights(state: kd.train.TrainState, save_dir: str):
    # Extract only the keys that belong to Titans (memory, memory_gate)
    original_params, titans_params = titans_tree_utils.split_titans_params(state.params)
    
    checkpointer = ocp.StandardCheckpointer()
    checkpointer.save(os.path.abspath(save_dir), titans_params, force=True)
    print(f"Saved ONLY Titans weights to {save_dir}")

def load_titans_weights(state: kd.train.TrainState, load_dir: str) -> kd.train.TrainState:
    checkpointer = ocp.StandardCheckpointer()
    titans_params = checkpointer.restore(os.path.abspath(load_dir))
    
    # Split the current state, swap out the titans portion, and merge back
    original_params, _ = titans_tree_utils.split_titans_params(state.params)
    merged_params = titans_tree_utils.merge_titans_params(original_params, titans_params)
    
    return state.replace(params=merged_params)

# Usage:
# save_titans_weights(state, "./saved_titans_delta")
# state = load_titans_weights(state, "./saved_titans_delta")

## 4. Dialogue Inference (Test-Time Dynamic Memory)
During inference, the model passes a `cache` containing both the standard Attention KV-cache AND the Titans Neural Memory State. The model automatically updates its internal state.

In [ ]:
def dialogue_turn(params, cache, user_input_tokens):
    # 1. Forward pass returns updated cache (including updated Titans memory)
    output = model.apply({'params': params}, tokens=user_input_tokens, cache=cache)
    
    logits = output.logits
    new_cache = output.cache

    # 2. Logic to extract the response token goes here
    return logits, new_cache

# Initialize empty state
print("Initializing Cache...")
# cache = model.init_cache(batch_size=1, dtype=jnp.bfloat16, cache_length=512)

print("Simulation: User enters a turn...")
# user_tokens = jnp.array([[1, 2, 3, 4]])
# logits, cache = dialogue_turn(state.params, cache, user_tokens)
# print("Memory state updated automatically in cache.")